# Form features from past games only

`nba_predictor.features` gives each team, before each game, its games played, win % and average margin so far this season, its average margin over the last 10 games, and offensive and defensive rating (points per 100 possessions) so far.

Everything here uses the **training seasons only** (2014-15 to 2019-20). Validation and test stay unseen until the models are compared.

In [1]:
import os
from pathlib import Path

os.environ.setdefault("NBA_DATA_DIR", "../data")

import pandas as pd
from sklearn.metrics import roc_auc_score

from nba_predictor import features

data = Path(os.environ["NBA_DATA_DIR"]) / "processed"
games = pd.read_parquet(data / "games.parquet")
feats = pd.read_parquet(data / "features.parquet")
train = feats[feats["season"] <= "2019-20"]
len(train)

7209

## Leakage check on the real data

The unit tests do this on made-up schedules. Here it runs on the real games: cut the data at a date, rebuild the features, and every game before the cut must come out identical.

In [2]:
full = features.game_features(games).set_index("game_id")
cols = [f"{s}_{c}" for s in ("home", "away") for c in features.TEAM_FEATURES]
for cut in ["2016-01-15", "2020-03-11", "2023-02-01"]:
    part = features.game_features(games[games["date"] < cut]).set_index("game_id")
    pd.testing.assert_frame_equal(part[cols], full.loc[part.index, cols])
    print(f"cut at {cut}: {len(part):,} earlier games unchanged")

cut at 2016-01-15: 1,822 earlier games unchanged


cut at 2020-03-11: 7,117 earlier games unchanged


cut at 2023-02-01: 10,287 earlier games unchanged


## How much each home-minus-away difference says on its own

In [3]:
rows = []
for col in features.TEAM_FEATURES:
    x = train[f"diff_{col}"]
    if col == "def_rtg":
        x = -x  # a lower defensive rating is better
    rows.append({"feature": col, "auc": roc_auc_score(train["home_win"], x)})
pd.DataFrame(rows).sort_values("auc", ascending=False).round(3)

,feature,auc
2,margin_avg,0.697
1,win_pct,0.693
3,form_margin,0.678
4,off_rtg,0.660
5,def_rtg,0.644
0,games_played,0.467


In [4]:
first = train[["home_games_played", "away_games_played"]].min(axis=1)
bucket = pd.cut(first, [-1, 0, 9, 29, 100], labels=["first game", "1-9", "10-29", "30+"])
pd.DataFrame(
    {
        "games": train.groupby(bucket, observed=True).size(),
        "auc of margin_avg": train.groupby(bucket, observed=True).apply(
            lambda g: roc_auc_score(g["home_win"], g["diff_margin_avg"])
        ),
    }
).round(3)

,games,auc of margin_avg
first game,98,0.516
1-9,837,0.636
10-29,1824,0.712
30+,4450,0.709


In [5]:
train_games = games[games["season"] <= "2019-20"]
pd.Series(
    {
        w: roc_auc_score(
            train_games["home_win"],
            features.game_features(train_games, window=w)["diff_form_margin"],
        )
        for w in (5, 10, 20)
    },
    name="auc of form_margin by window",
).round(3)

5     0.652
10    0.678
20    0.689
Name: auc of form_margin by window, dtype: float64

## Findings

1. **Average margin so far is the best single feature** (AUC 0.697), just ahead of win % (0.693). The ratings say less on their own (0.66 offence, 0.64 defence), probably because each covers only half of what margin does.
2. **Features are weak early in the season.** With 1 to 9 games played, margin gives 0.64. From 10 games on it's 0.71. On opening night it's a coin flip (98 games, placeholders only). Carrying a rating over from last season, as Elo does (day 5), should help most here.
3. **"Recent form" mostly repeats the season average.** A longer window is better (5 games 0.652, 10 games 0.678, 20 games 0.689), approaching the full-season figure. The 10-game window stays for now. Day 8's ablation tests whether it adds anything at all beyond the season average.
4. **`diff_games_played` is close to noise** (0.47). It's kept because it tells the model when the other features are placeholders, not as a predictor.